# 02 — Trening modeli

Trenujemy XGBoost, LightGBM, RF, EBM i porównujemy.

Wymagania: uruchomione `scripts/02_preprocess.py` (train/val/test.parquet w `data/processed/`).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np

from src.models import (
    XGBoostTriageModel, LightGBMTriageModel, 
    RandomForestTriageModel, EBMTriageModel
)
from src.data.preprocessing import build_feature_groups, split_features
from src.evaluation.metrics import full_evaluation
from src.evaluation.visualizations import plot_confusion_matrix, plot_roc_curves
from src.utils.config import TRAIN_PARQUET, VAL_PARQUET, TEST_PARQUET

In [ ]:
df_train = pd.read_parquet(TRAIN_PARQUET)
df_val = pd.read_parquet(VAL_PARQUET)
df_test = pd.read_parquet(TEST_PARQUET)

groups = build_feature_groups(df_train)
X_train, y_train, _ = split_features(df_train, groups, feature_set='triage_only')
X_val, y_val, _ = split_features(df_val, groups, feature_set='triage_only')
X_test, y_test, _ = split_features(df_test, groups, feature_set='triage_only')

print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')

## XGBoost

In [ ]:
model_xgb = XGBoostTriageModel()
model_xgb.fit(
    X_train, y_train,
    X_val=X_val, y_val=y_val,
    sample_weight_strategy='custom',
    feature_set='triage_only',
)

In [ ]:
y_pred = model_xgb.predict(X_test)
y_proba = model_xgb.predict_proba(X_test)
metrics_xgb = full_evaluation(y_test, y_pred, y_proba, prefix='test')

In [ ]:
plot_confusion_matrix(y_test, y_pred, show=True)
plot_roc_curves(y_test, y_proba, show=True)

## LightGBM

In [ ]:
model_lgbm = LightGBMTriageModel()
model_lgbm.fit(X_train, y_train, X_val=X_val, y_val=y_val)
metrics_lgbm = full_evaluation(
    y_test, model_lgbm.predict(X_test), model_lgbm.predict_proba(X_test), prefix='test',
)

## Porównanie modeli

In [ ]:
comparison = pd.DataFrame({
    'XGBoost': metrics_xgb,
    'LightGBM': metrics_lgbm,
}).T
comparison[['test_quadratic_weighted_kappa', 'test_undertriage_rate', 'test_auc_macro', 'test_f1_macro']]